# sfig25 — Modality Importance Radar Chart (Supplementary Fig. S-25)

Polar/radar chart: each axis = |ΔAUROC| when that modality is removed.
One polygon per task. 5 spokes: No BAS, No RESP, No EKG, Cardio only, BAS only.

**Data**: `results/tables/table6_modality.csv`.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# ── Workspace root ────────────────────────────────────────────────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final_npj"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Main utils first (must come before explore to avoid shadowing utils.style)
_nb_dir = PAPER_FIGURES / "notebooks_npj"
sys.path.insert(0, str(_nb_dir))
# Explore panel functions — add utils/ subdir directly, not the parent package
_explore_utils = PAPER_FIGURES / "explore" / "notebooks" / "utils"
sys.path.insert(1, str(_explore_utils))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS,
    TASK_LABEL, FONT_BASE, FONT_LABEL, FONT_TITLE, CTX_ORDER,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from data_explore import load_modality_table   # only in explore utils dir
from utils import panels
import panels_explore as xp   # from explore utils dir, no package conflict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

WORKSPACE_ROOT : /Users/boshra/NSRR-workspace
final_results/ : ✓ found


In [3]:
# ── Load modality table ───────────────────────────────────────────────────────
mod_df = load_modality_table(NSRR_TOOLS)
print(mod_df.to_string())
print("\nColumns:", mod_df.columns.tolist())

                   Task Context  N_test   Full  No BAS  Δ(No BAS)  No RESP  Δ(No RESP)  No EKG  Δ(No EKG)  Cardio only  Δ(Cardio only)  BAS only  Δ(BAS only)
0                   Sex    120m    1430  0.872   0.803     -0.069    0.861      -0.011   0.799     -0.074        0.800          -0.072     0.781       -0.092
1  Sleep apnea (AHI≥15)    120m    2054  0.832   0.792     -0.040    0.775      -0.057   0.794     -0.038        0.766          -0.066     0.729       -0.103
2      Sleep efficiency    120m    2020  0.778   0.695     -0.083    0.776      -0.003   0.765     -0.013        0.667          -0.111     0.773       -0.005
3             Age group    120m    1859  0.893   0.847     -0.046    0.885      -0.008   0.876     -0.017        0.824          -0.069     0.858       -0.035
4           BMI (obese)     40m    1856  0.756   0.721     -0.035    0.766       0.010   0.751     -0.005        0.675          -0.081     0.742       -0.014

Columns: ['Task', 'Context', 'N_test', 'Full', 'No 

In [ ]:
# Task names in the table vs internal keys — the panel resolves these itself
# (panels_explore._MODALITY_ROW_KEY); a bare label prefix is not unique, since
# "slee" matches both "Sleep apnea" and "Sleep efficiency".
TASKS = MAIN_TASKS

fig = plt.figure(figsize=(5.5, 5.5))
ax = fig.add_subplot(111, projection="polar")

handles, labels = xp.modality_radar_panel(ax, mod_df, tasks=TASKS)

# Legend comes from what was actually drawn, not from TASKS: building it from
# TASKS is what previously let sleep efficiency appear in the legend while no
# polygon of its own was on the chart.
ax.legend(handles, labels,
          loc="upper left", bbox_to_anchor=(1.0, 1.0),
          fontsize=7, frameon=False)

_missing = [TASK_LABEL.get(t, t) for t in TASKS
            if TASK_LABEL.get(t, t) not in labels]
assert not _missing, f"tasks in TASKS but not drawn: {_missing}"
print(f"drawn polygons ({len(labels)}): {labels}")

# ax.set_title("Modality importance: |ΔAUROC| when modality removed",
            #  fontsize=FONT_TITLE, pad=15)
fig.tight_layout()
plt.show()


In [ ]:
# ── Run when figure looks good ────────────────────────────────────────────────
save_figure(fig, FINAL_OUT, "sfig25_modality_radar")
import shutil
shutil.copy(FINAL_OUT / "sfig25_modality_radar.pdf",
            WORKSPACE_ROOT / "npj_digital_medicine_submission" / "figures" / "sfig25_modality_radar.pdf")
print("Saved + copied → npj_digital_medicine_submission/figures/sfig25_modality_radar.pdf")

  saved → /Users/boshra/NSRR-workspace/NSRR-tools/results/paper_figures/final/sfig15_modality_radar.pdf
Saved + copied → TBME_submission/sfig15_modality_radar.pdf
